# Correr todo — las 6 verticales en una sola ejecución

Corre lavanderías, hoteles, panadería, gastro, bodegas y PM, una detrás de otra. Los insumos
que comparten (locales, Equifax, maestro de mercado) se cargan una sola vez.

Para correr solo una vertical, usa `notebooks/orquestador.ipynb` (cliente) o
`notebooks/orquestador_pm.ipynb` (PM) en su lugar — son más simples de leer paso a paso.

In [1]:
# Rutas de los insumos locales (ajusta a tu entorno — ver README, sección "data_potential")
RUTA_LOCALES = "../data_potential/input/locales.csv"
RUTA_EQUIFAX = "../data_potential/input/equifax.csv"
RUTA_URBANICIDAD = "../data_potential/input/urbanicidad.csv"
RUTA_MAESTRO_MERCADO = "../data_potential/input/Maestro_de_mercado.xlsx"
RUTA_CLUSTER_MERCADO_ID = "../data_potential/input/cluster_mercado_id.csv"
RUTA_SALIDA = "../data_potential/output"
RUTA_METRICAS = "../data_potential/output/metricas_modelos.csv"
RUTA_PDM_1 = "../data_potential/input/pdm_1.csv"
RUTA_PDM_2 = "../data_potential/input/pdm_2.csv"
RUTA_TOP15 = "../data_potential/input/top15.csv"


In [2]:
import sys, time, warnings
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import get_vertical, load_config
from src.data import fetch_clientes, fetch_puntos
from src.features import agregar_features_cliente
from src.equifax import prepare_equifax_sum
from src.pm_mercado import assign_mercado_id, aggregate_mercado, map_segmento_pm, reference_rows
from src.model import fit_potential
from src.decision import apply_decision
from src.assemble import assemble_output
from src.export import compute_feature_mean, write_excel
from src.metrics_log import log_metrics
from src.pdm import construir_pdm_final, map_segmento_pdm

VERTICALES_CLIENTE = ["lavanderias", "hoteles", "panaderia", "gastro", "bodegas"]

## Insumos compartidos (una sola vez)

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project="acpe-dev-uc-ml")

t0 = time.time()
df_puntos = fetch_puntos(client)
locales = pd.read_csv(RUTA_LOCALES, sep=";", encoding="utf-8-sig")
equifax_sum = prepare_equifax_sum(pd.read_csv(RUTA_EQUIFAX), pd.read_csv(RUTA_URBANICIDAD))
maestro = pd.read_excel(RUTA_MAESTRO_MERCADO, usecols=range(14)).dropna(axis=1, how="all")
cluster_fallback = pd.read_csv(RUTA_CLUSTER_MERCADO_ID)
diccionario = pd.read_csv(Path("..") / "configs" / "diccionario_encuesta.csv")
print(f"✓ Insumos compartidos listos ({time.time() - t0:.0f}s): "
      f"{len(df_puntos):,} puntos, {len(locales):,} locales, {len(equifax_sum):,} manzanas Equifax")


C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ Insumos compartidos listos (46s): 123,538 puntos, 1,305 locales, 92,121 manzanas Equifax


## Funciones (una por tipo de vertical; repiten los pasos 1 a 6 de cada orquestador)

In [4]:
def run_vertical_cliente(vertical: str) -> Path:
    cfg = get_vertical(vertical)
    m, d, e = cfg["modelo"], cfg["decision"], cfg["export"]

    df_clientes = fetch_clientes(vertical, client)
    df_features, info = agregar_features_cliente(df_clientes, df_puntos, locales, equifax_sum)
    print(f"[{vertical}] {info}")
    df_features = df_features.rename(columns={"cod_cliente_dex": "Cod Cliente DEX"})

    df_modelo = df_features.copy()
    if m["target_map"]:
        df_modelo["segmento_trx"] = df_modelo["segmento_trx"].map(m["target_map"])
    resultado = fit_potential(
        df_modelo, m["features"], m["categoricas"], cfg["class_order"], m["lgbm"],
        id_col="Cod Cliente DEX", cluster_col=m["cluster_col"], split_valid=m["split_validacion"],
        optuna_trials=m["optuna"]["n_trials"] if m["optuna"]["activo"] else None,
    )
    log_metrics(vertical, resultado, RUTA_METRICAS)
    estimaciones = apply_decision(resultado.estimates, d["regla"], cfg["class_order"], d["high"], d["medium"])
    salida = assemble_output(estimaciones, df_features, m["features"], id_col="Cod Cliente DEX")

    fm = cfg["feature_mean"]
    feature_mean = compute_feature_mean(
        salida, cfg["class_order"], fm["rmv_feats"], fm["modo"],
        features=[f for f in m["features"] if f not in set(fm["rmv_feats"])] if fm["modo"] == "features" else None,
        reference_df=df_features,
    )
    return _guardar(cfg, e, salida, feature_mean, estimaciones["flag_potential"], vertical)


def run_pm() -> dict:
    cfg = get_vertical("pm_mercado")
    m, d, e = cfg["modelo"], cfg["decision"], cfg["export"]

    df_clientes = fetch_clientes("pm_mercado", client)
    df_features, info = agregar_features_cliente(df_clientes, df_puntos, locales, equifax_sum)
    print(f"[pm_mercado] {info}")
    df_features["mercado_id"] = assign_mercado_id(df_features, maestro, cluster_fallback, id_col="cod_cliente_dex")
    df_features = df_features.rename(columns={"cod_cliente_dex": "Cod Cliente DEX"})
    df_con_mercado = df_features.dropna(subset=["mercado_id"]).copy()

    ub_mercado = aggregate_mercado(df_con_mercado, equifax_sum, equifax_radio_km=cfg["agregacion"]["equifax_radio_km"])
    ub_mercado = map_segmento_pm(ub_mercado)

    resultado = fit_potential(
        ub_mercado, m["features"], m["categoricas"], cfg["class_order"], m["lgbm"],
        id_col="mercado_id", cluster_col=m["cluster_col"], split_valid=m["split_validacion"],
        optuna_trials=m["optuna"]["n_trials"] if m["optuna"]["activo"] else None,
    )
    log_metrics("pm_mercado", resultado, RUTA_METRICAS)
    estimaciones = apply_decision(resultado.estimates, d["regla"], cfg["class_order"], d["high"], d["medium"])
    salida_estimada = assemble_output(estimaciones, ub_mercado, m["features"], id_col="mercado_id", geo_cols=("des_ciudad",))
    ref = reference_rows(ub_mercado, cfg["class_order"], id_col="mercado_id", features=m["features"])
    salida = pd.concat([salida_estimada, ref], ignore_index=True)

    fm = cfg["feature_mean"]
    feature_mean = compute_feature_mean(salida, cfg["class_order"], fm["rmv_feats"], fm["modo"])
    ids_mercado = df_con_mercado[["Cod Cliente DEX", "mercado_id"]].drop_duplicates()

    fecha = time.strftime("%Y%m%d")
    ruta_pm = Path(RUTA_SALIDA) / f"{fecha}_{cfg['salida']}.xlsx"
    ruta_pm.parent.mkdir(parents=True, exist_ok=True)
    write_excel(
        str(ruta_pm),
        {e["hoja_principal"]: salida, "IDs mercado": ids_mercado,
         "Diccionario Encuesta": diccionario, "Feature mean": feature_mean},
        hojas_con_formato=[e["hoja_principal"]] if e["con_formato"] else [],
    )
    print(f"✓ [pm_mercado] {int(estimaciones['flag_potential'].sum()):,} potenciales de {len(estimaciones):,} — {ruta_pm}")

    # ---- Paso final PDM, directo en memoria (sin releer el Excel de PM) ----
    segmento_por_cliente = df_features[["Cod Cliente DEX", "des_segmento_estrategico"]].drop_duplicates("Cod Cliente DEX")
    segmento_por_cliente["segmento"] = map_segmento_pdm(segmento_por_cliente["des_segmento_estrategico"])
    segmento_por_cliente = segmento_por_cliente[["Cod Cliente DEX", "segmento"]]

    potencial_pm = estimaciones.loc[estimaciones["flag_potential"] == 1.0, ["mercado_id"]].assign(flag_potential=1.0)

    pdm_1 = pd.read_csv(RUTA_PDM_1)
    pdm_2 = pd.read_csv(RUTA_PDM_2)
    top15 = pd.read_csv(RUTA_TOP15)
    salidas_pdm = construir_pdm_final(
        pdm_1, pdm_2, top15, ids_mercado, segmento_por_cliente, potencial_pm,
        factor_umbral=cfg["pdm"]["factor_umbral"],
    )
    for hoja, df in salidas_pdm.items():
        print(f"  [PDM/{hoja}] {len(df):,} clientes — {df['Potencial'].value_counts().to_dict()}")

    ruta_pdm = Path(RUTA_SALIDA) / f"{fecha}_Potencial_final_PDM.xlsx"
    write_excel(str(ruta_pdm), {"Abarroteros": salidas_pdm["Abarroteros"], "Lavanderos": salidas_pdm["Lavanderos"],
                                "Impulso": salidas_pdm["Impulso"]}, hojas_con_formato=[])
    print(f"✓ [PDM] Guardado en {ruta_pdm}")

    return {"pm": ruta_pm, "pdm": ruta_pdm}


def _guardar(cfg, e, salida, feature_mean, flag_potential, nombre) -> Path:
    fecha = time.strftime("%Y%m%d")
    ruta = Path(RUTA_SALIDA) / f"{fecha}_{cfg['salida']}.xlsx"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    write_excel(
        str(ruta),
        {e["hoja_principal"]: salida, "Diccionario Encuesta": diccionario, "Feature mean": feature_mean},
        hojas_con_formato=[e["hoja_principal"]] if e["con_formato"] else [],
    )
    print(f"✓ [{nombre}] {int(flag_potential.sum()):,} potenciales de {len(flag_potential):,} — {ruta}")
    return ruta

## Correr las 6

In [5]:
with warnings.catch_warnings():
    warnings.simplefilter("default")
    rutas = {v: run_vertical_cliente(v) for v in VERTICALES_CLIENTE}
    rutas["pm_mercado"] = run_pm()  # {"pm": ruta_excel_pm, "pdm": ruta_excel_pdm}

print("\nListo:")
for v, r in rutas.items():
    print(f"  {v:12s} -> {r}")

C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[lavanderias] {'clientes_originales': 2477, 'sin_coordenadas_validas': 30, 'con_coordenadas_validas': 2447, 'coordenadas_intercambiadas': 9}
✓ [lavanderias] 517 potenciales de 1,767 — ..\data_potential\output\20261020_potential_estimations_lavanderia_nuevo.xlsx


C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[hoteles] {'clientes_originales': 3089, 'sin_coordenadas_validas': 31, 'con_coordenadas_validas': 3058, 'coordenadas_intercambiadas': 12}
✓ [hoteles] 372 potenciales de 2,474 — ..\data_potential\output\20261020_potential_estimations_hotel_nuevo.xlsx


C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[panaderia] {'clientes_originales': 10119, 'sin_coordenadas_validas': 62, 'con_coordenadas_validas': 10057, 'coordenadas_intercambiadas': 51}
✓ [panaderia] 2,230 potenciales de 7,043 — ..\data_potential\output\20261020_potential_estimations_panaderia_nuevo.xlsx


... (4 outputs left out)

C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[pm_mercado] {'clientes_originales': 29422, 'sin_coordenadas_validas': 90, 'con_coordenadas_validas': 29332, 'coordenadas_intercambiadas': 14}
✓ [pm_mercado] 461 potenciales de 1,895 — ..\data_potential\output\20261020_potential_estimations_market_level_mean_seg_pm.xlsx


C:\Users\asucasacav\AppData\Local\Temp\ipykernel_9572\4242463219.py:78: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  pdm_1 = pd.read_csv(RUTA_PDM_1)


  [PDM/Abarroteros] 694 clientes — {'Medium': 502, 'No Potencial': 166, 'Low': 26}
  [PDM/Lavanderos] 92 clientes — {'Medium': 64, 'No Potencial': 23, 'Low': 5}
  [PDM/Impulso] 26 clientes — {'No Potencial': 13, 'Medium': 11, 'Low': 2}
✓ [PDM] Guardado en ..\data_potential\output\20261020_Potencial_final_PDM.xlsx

Listo:
  lavanderias  -> ..\data_potential\output\20261020_potential_estimations_lavanderia_nuevo.xlsx
  hoteles      -> ..\data_potential\output\20261020_potential_estimations_hotel_nuevo.xlsx
  panaderia    -> ..\data_potential\output\20261020_potential_estimations_panaderia_nuevo.xlsx
  gastro       -> ..\data_potential\output\20261020_potential_estimations_gastro_nuevo.xlsx
  bodegas      -> ..\data_potential\output\20261020_potential_estimations_bodegas_nuevo.xlsx
  pm_mercado   -> {'pm': WindowsPath('../data_potential/output/20261020_potential_estimations_market_level_mean_seg_pm.xlsx'), 'pdm': WindowsPath('../data_potential/output/20261020_Potencial_final_PDM.xlsx')}
